In [0]:
# Weekly guard: in the scheduled job, run only on Monday's morning run (Dubai time).
# For a manual run on another day, set force_weekly = true.
from datetime import datetime
from zoneinfo import ZoneInfo

dbutils.widgets.dropdown("force_weekly", "false", ["false", "true"])
_now = datetime.now(ZoneInfo("Asia/Dubai"))
if not ((_now.weekday() == 0 and _now.hour < 12) or dbutils.widgets.get("force_weekly") == "true"):
    dbutils.notebook.exit(f"Skipped: weekly model, runs Monday mornings (now {_now:%a %H:%M} Dubai)")

In [0]:
import time
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

t0 = time.time()
run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

w = spark.table("oil_shock.gold.fct_us_weekly_balance").toPandas()
w["week_ending"] = pd.to_datetime(w["week_ending"])
w = w.sort_values("week_ending").reset_index(drop=True)
next_week = w["week_ending"].max() + pd.Timedelta(days=7)
w = pd.concat([w, pd.DataFrame({"week_ending": [next_week]})], ignore_index=True)   # row to nowcast

y = "crude_stocks_change_kbbl"
w["gas_chg"] = w["gasoline_stocks_kbbl"].diff()
w["dist_chg"] = w["distillate_stocks_kbbl"].diff()
for c in [y, "gas_chg", "dist_chg", "refinery_utilization_pct", "crude_net_imports_kbd",
          "crude_production_kbd", "spr_change_kbbl"]:
    w[f"{c}_l1"] = w[c].shift(1)
w["chg_l2"] = w[y].shift(2)
w["chg_4w"] = w[y].shift(1).rolling(4).mean()
woy = w["week_ending"].dt.isocalendar().week.astype(int).values
w["woy_sin"], w["woy_cos"] = np.sin(2 * np.pi * woy / 52), np.cos(2 * np.pi * woy / 52)

# Seasonal benchmark: same week (+/-1) in the previous 5 years
yr = w["week_ending"].dt.year.values
vals = w[y].values
seas = np.full(len(w), np.nan)
for i in range(len(w)):
    m = (yr < yr[i]) & (yr >= yr[i] - 5) & (np.abs(woy - woy[i]) <= 1) & ~np.isnan(vals)
    if m.sum() >= 5:
        seas[i] = vals[m].mean()
w["seasonal"] = seas

# Brent-WTI spread as of Tuesday before the Wednesday release
px = spark.table("oil_shock.gold.fct_model_features_daily").select("price_date", "brent_wti_spread").toPandas()
px["price_date"] = pd.to_datetime(px["price_date"])
w["tuesday"] = w["week_ending"] + pd.Timedelta(days=4)
w = pd.merge_asof(w.sort_values("tuesday"), px.sort_values("price_date").dropna(),
                  left_on="tuesday", right_on="price_date", direction="backward").sort_values("week_ending")

FEATS = [f"{y}_l1", "chg_l2", "chg_4w", "gas_chg_l1", "dist_chg_l1", "refinery_utilization_pct_l1",
         "crude_net_imports_kbd_l1", "crude_production_kbd_l1", "spr_change_kbbl_l1",
         "woy_sin", "woy_cos", "seasonal", "brent_wti_spread"]
data = w.dropna(subset=FEATS).reset_index(drop=True)
hist = data[data[y].notna()]
print(f"Weeks with features: {len(hist)} | {hist['week_ending'].min().date()} to {hist['week_ending'].max().date()} "
      f"| nowcast week: {next_week.date()}  ({time.time() - t0:.0f}s)")

def make_models():
    return {"ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))),
            "gbm": HistGradientBoostingRegressor(max_depth=2, learning_rate=0.05, max_iter=150,
                                                 min_samples_leaf=20, random_state=42)}

test = hist[hist["week_ending"] >= "2019-01-01"]
preds, models, refits = [], None, 0
for i, (_, row) in enumerate(test.iterrows()):
    if i % 4 == 0:
        train = hist[hist["week_ending"] < row["week_ending"]]
        models = make_models()
        for mdl in models.values():
            mdl.fit(train[FEATS], train[y])
        refits += 1
        if refits % 20 == 0:
            print(f"  refit {refits}, week {row['week_ending'].date()}  ({time.time() - t0:.0f}s)")
    x = row[FEATS].to_frame().T.astype(float)
    preds.append({"week_ending": row["week_ending"], "actual": row[y], "zero": 0.0,
                  "last_week": row[f"{y}_l1"], "seasonal": row["seasonal"],
                  "ridge": models["ridge"].predict(x)[0], "gbm": models["gbm"].predict(x)[0]})
bt = pd.DataFrame(preds)
bt["period"] = np.where(bt["week_ending"] >= "2026-03-01", "war", "pre_war")

METHODS = ["zero", "last_week", "seasonal", "ridge", "gbm"]
rows = []
for per, g in list(bt.groupby("period")) + [("all", bt)]:
    for mth in METHODS:
        err = g[mth] - g["actual"]
        dir_ok = np.sign(g[mth]) == np.sign(g["actual"]) if mth != "zero" else pd.Series([np.nan] * len(g))
        rows.append({"period": per, "method": mth, "weeks": len(g),
                     "mae_kbbl": round(err.abs().mean()), "direction_hit_pct": round(100 * dir_ok.mean(), 1)})
summary = pd.DataFrame(rows)

# Nowcast for the next release with models fit on all history
final = make_models()
for mdl in final.values():
    mdl.fit(hist[FEATS], hist[y])
xn = data.loc[data["week_ending"] == next_week, FEATS].astype(float)
nowcast = {"week_ending": next_week.date(), "seasonal": float(xn["seasonal"].iloc[0]),
           "ridge": float(final["ridge"].predict(xn)[0]), "gbm": float(final["gbm"].predict(xn)[0])}

# Save and log
out_bt = bt.copy(); out_bt["run_at_utc"] = run_stamp
out_sum = summary.copy(); out_sum["run_at_utc"] = run_stamp
out_now = pd.DataFrame([nowcast]); out_now["run_at_utc"] = run_stamp
for name, out in {"inventory_nowcast_backtest": out_bt, "inventory_nowcast_summary": out_sum,
                  "inventory_nowcast_current": out_now}.items():
    spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"oil_shock.ml.{name}")

allp = summary[summary["period"] == "all"].set_index("method")
best_model = allp.loc[["ridge", "gbm"], "mae_kbbl"].idxmin()
passes = allp.loc[best_model, "mae_kbbl"] < allp.loc["seasonal", "mae_kbbl"]
mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-inventory-nowcast")
with mlflow.start_run(run_name=f"inventory_nowcast {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/43_inventory_nowcast", "project": "oil-shock-2026",
                     "author": "Zia Malik", "kill_rule": "best model must beat seasonal MAE",
                     "kill_rule_outcome": "passed" if passes else "TRIGGERED: use seasonal benchmark"})
    mlflow.log_params({"target": "EIA weekly US commercial crude stock change (kbbl)",
                       "backtest": "weekly from 2019, refit every 4 weeks, expanding window",
                       "features": ", ".join(FEATS)})
    for _, r in summary.iterrows():
        mlflow.log_metric(f"{r['period']}_{r['method']}_mae", float(r["mae_kbbl"]))

# ---------- Compact summary to paste ----------
print(f"\nDone in {time.time() - t0:.0f}s | refits: {refits} | test weeks: {len(bt)}")
print(summary.pivot(index="method", columns="period", values="mae_kbbl").to_string())
print("Direction hit % (all):", allp["direction_hit_pct"].drop("zero").to_dict())
print(f"Kill rule: best model {best_model} {allp.loc[best_model, 'mae_kbbl']} vs seasonal "
      f"{allp.loc['seasonal', 'mae_kbbl']} kbbl -> {'PASSES' if passes else 'FAILS'}")
print("Nowcast for week ending", nowcast["week_ending"], {k: round(v) for k, v in nowcast.items() if k != "week_ending"})